# Kairos: planificación de tareas como PSR
Proyecto final de Inteligencia Artificial.

Kairos decide en qué bloque de 30 minutos empieza cada tarea para maximizar la suma de prioridades de las tareas programadas. El notebook va en cuatro fases, cada una con sus bloques de código:

1. Backtracking puro (fuerza bruta)
2. Heurísticas VMR, VMRe y comprobación directa
3. Caso realista con bloques fijos y poda optimista
4. Prueba de estrés y alertas

## Fase 1: backtracking puro (fuerza bruta)
Primero se modela Kairos como un PSR: cada tarea es una variable, su dominio son los bloques de 30 minutos donde puede empezar más el valor Ø (no programada), y las restricciones son R2 (sin cruces y con descanso) y R4 (tope diario). La búsqueda recorre las tareas en un orden fijo y cuenta cada intento como un nodo. Esta versión es la línea base para comparar.

In [1]:
BLOQUES_POR_DIA = 8
DIAS = ["Lunes", "Martes"]
NO_PROGRAMADA = None

### Variables y dominios

In [2]:
TAREAS = {
    "A": {"dur": 6, "prio": 5, "vence": (1, 8)},
    "B": {"dur": 6, "prio": 3, "vence": (0, 8)},
    "C": {"dur": 2, "prio": 2, "vence": (1, 8)},
}
ORDEN = ["A", "B", "C"]


def hora(bloque):
    m = 18 * 60 + bloque * 30
    return f"{m // 60:02d}:{m % 60:02d}"


def construir_dominios():
    """Dominio de cada tarea: (día, bloque_inicio) válidos + 'Ø' al final."""
    dominios = {}
    for t in ORDEN:
        dur = TAREAS[t]["dur"]
        dia_v, blq_v = TAREAS[t]["vence"]
        dom = []
        for d in range(len(DIAS)):
            for ini in range(BLOQUES_POR_DIA):
                fin = ini + dur
                if fin > BLOQUES_POR_DIA:
                    continue
                if d > dia_v or (d == dia_v and fin > blq_v):
                    continue
                dom.append((d, ini))
        dom.append(NO_PROGRAMADA)
        dominios[t] = dom
    return dominios

### Restricciones

In [3]:
def consistente(asignacion, tarea, valor):
    """Verifica R2 y R4 de la nueva asignación contra las ya hechas."""
    if valor is NO_PROGRAMADA:
        return True
    dia, ini = valor
    fin = ini + TAREAS[tarea]["dur"]
    bloques_dia = TAREAS[tarea]["dur"]

    for otra, v in asignacion.items():
        if v is NO_PROGRAMADA:
            continue
        d2, ini2 = v
        if d2 != dia:
            continue
        fin2 = ini2 + TAREAS[otra]["dur"]
        bloques_dia += TAREAS[otra]["dur"]
        if not (ini >= fin2 + 1 or ini2 >= fin + 1):
            return False

    if bloques_dia > 8:
        return False
    return True

### Backtracking con contador de nodos

In [4]:
class Buscador:
    def __init__(self):
        self.dominios = construir_dominios()
        self.nodos = 0
        self.mejor_prio = -1
        self.mejor_asig = None

    def prioridad(self, asig):
        return sum(TAREAS[t]["prio"] for t, v in asig.items() if v is not NO_PROGRAMADA)

    def backtrack(self, asignacion, i=0):
        if i == len(ORDEN):
            p = self.prioridad(asignacion)
            if p > self.mejor_prio:
                self.mejor_prio = p
                self.mejor_asig = dict(asignacion)
            return
        tarea = ORDEN[i]
        for valor in self.dominios[tarea]:
            self.nodos += 1
            if consistente(asignacion, tarea, valor):
                asignacion[tarea] = valor
                self.backtrack(asignacion, i + 1)
                del asignacion[tarea]

### Resultados

In [5]:
def mostrar(asig, prio):
    print(f"Mejor horario (prioridad total = {prio}):")
    for t in ORDEN:
        v = asig[t]
        if v is NO_PROGRAMADA:
            print(f"  Tarea {t}: Ø (no programada)")
        else:
            d, ini = v
            fin = ini + TAREAS[t]["dur"]
            print(f"  Tarea {t}: {DIAS[d]} {hora(ini)} - {hora(fin)}")


if __name__ == "__main__":
    b = Buscador()
    for t in ORDEN:
        print(f"Dominio {t}: {len(b.dominios[t])} valores")
    b.backtrack({})
    print()
    mostrar(b.mejor_asig, b.mejor_prio)
    print(f"\nNodos evaluados: {b.nodos}")

Dominio A: 7 valores
Dominio B: 4 valores
Dominio C: 15 valores

Mejor horario (prioridad total = 8):
  Tarea A: Martes 18:00 - 21:00
  Tarea B: Lunes 18:00 - 21:00
  Tarea C: Ø (no programada)

Nodos evaluados: 320


**Resultado:** el mejor horario tiene prioridad 8 y la fuerza bruta evalúa 320 nodos para encontrarlo.

## Fase 2: heurísticas VMR, VMRe y comprobación directa
Sobre el mismo problema se agregan tres heurísticas: VMR elige primero la tarea con menos bloques posibles, VMRe ordena los valores según cuántas opciones dejan a las demás tareas, y la comprobación directa (CD) recorta los dominios después de cada asignación. Al final se compara contra la fuerza bruta.

In [6]:
import matplotlib.pyplot as plt

### Datos del problema (igual que en la versión pura)

In [7]:
BLOQUES_POR_DIA = 8
DIAS = ["Lunes", "Martes"]
NO_PROGRAMADA = None

TAREAS = {
    "A": {"dur": 6, "prio": 5, "vence": (1, 8)},
    "B": {"dur": 6, "prio": 3, "vence": (0, 8)},
    "C": {"dur": 2, "prio": 2, "vence": (1, 8)},
}
ORDEN = ["A", "B", "C"]

nodos_fuerza_bruta = None

### Utilidades del modelo

In [8]:
def hora(bloque):
    m = 18 * 60 + bloque * 30
    return f"{m // 60:02d}:{m % 60:02d}"


def dominios_iniciales():
    """Bloques de inicio legales (día, bloque) por tarea. 'Ø' NO se guarda aquí:
    siempre está disponible y se prueba de último."""
    dominios = {}
    for t, info in TAREAS.items():
        dia_v, blq_v = info["vence"]
        dom = []
        for d in range(len(DIAS)):
            for ini in range(BLOQUES_POR_DIA):
                fin = ini + info["dur"]
                if fin > BLOQUES_POR_DIA:
                    continue
                if d > dia_v or (d == dia_v and fin > blq_v):
                    continue
                dom.append((d, ini))
        dominios[t] = dom
    return dominios


def consistente(asignacion, tarea, valor):
    """R2 (sin cruce + 1 bloque de descanso) y R4 (máx. 4 h por día)."""
    if valor is NO_PROGRAMADA:
        return True
    dia, ini = valor
    fin = ini + TAREAS[tarea]["dur"]
    bloques_dia = TAREAS[tarea]["dur"]
    for otra, v in asignacion.items():
        if v is NO_PROGRAMADA or v[0] != dia:
            continue
        ini2 = v[1]
        fin2 = ini2 + TAREAS[otra]["dur"]
        bloques_dia += TAREAS[otra]["dur"]
        if not (ini >= fin2 + 1 or ini2 >= fin + 1):
            return False
    return bloques_dia <= BLOQUES_POR_DIA


def prioridad(asignacion):
    return sum(TAREAS[t]["prio"] for t, v in asignacion.items() if v is not NO_PROGRAMADA)

### Heurísticas

In [9]:
def comprobacion_directa(dominios, asignacion, tarea, valor):
    """CD: tras asignar `valor` a `tarea`, recorta de los dominios de las tareas
    no asignadas todos los bloques que se cruzan o que no respetan el descanso
    de 30 min (R2), o que rompen el tope diario (R4).
    `dominios` contiene SOLO tareas no asignadas. Devuelve los nuevos dominios."""
    nueva = dict(asignacion)
    nueva[tarea] = valor
    return {
        t: [v for v in dom if consistente(nueva, t, v)]
        for t, dom in dominios.items() if t != tarea
    }


def seleccionar_variable_vmr(dominios):
    """VMR: elige la tarea no asignada con MENOS bloques legales en su dominio actual.
    Si una tarea quedó con 0 bloques (solo puede ser 'Ø'), sale primera: el
    fallo se detecta de inmediato. Empates: se respeta el orden de ORDEN."""
    return min(dominios, key=lambda t: (len(dominios[t]), ORDEN.index(t)))


def ordenar_valores_vmre(tarea, dominios, asignacion):
    """VMRe: ordena los valores de `tarea` de mayor a menor cantidad de bloques
    que dejan disponibles a las tareas restantes (se simula la CD de cada valor).
    'Ø' siempre se agrega de último."""
    puntuados = []
    for v in dominios[tarea]:
        restantes = comprobacion_directa(dominios, asignacion, tarea, v)
        libres = sum(len(d) for d in restantes.values())
        puntuados.append((libres, v))
    puntuados.sort(key=lambda x: -x[0])
    return [v for _, v in puntuados] + [NO_PROGRAMADA]

### Búsquedas

In [10]:
class BusquedaOptimizada:
    """Backtracking con VMR + VMRe + CD."""

    def __init__(self):
        self.nodos = 0
        self.vacios_detectados = 0
        self.mejor_prio = -1
        self.mejor_asig = None

    def resolver(self):
        self.backtrack({}, dominios_iniciales())

    def backtrack(self, asignacion, dominios):
        if not dominios:
            p = prioridad(asignacion)
            if p > self.mejor_prio:
                self.mejor_prio, self.mejor_asig = p, dict(asignacion)
            return

        tarea = seleccionar_variable_vmr(dominios)
        valores = ordenar_valores_vmre(tarea, dominios, asignacion)

        for valor in valores:
            self.nodos += 1
            if valor is NO_PROGRAMADA:
                nuevos = {t: d for t, d in dominios.items() if t != tarea}
            else:
                nuevos = comprobacion_directa(dominios, asignacion, tarea, valor)
                if any(len(d) == 0 for d in nuevos.values()):
                    self.vacios_detectados += 1
            asignacion[tarea] = valor
            self.backtrack(asignacion, nuevos)
            del asignacion[tarea]


def fuerza_bruta():
    """Versión pura (orden estático, sin heurísticas). Solo se usa si
    nodos_fuerza_bruta es None."""
    dom = {t: d + [NO_PROGRAMADA] for t, d in dominios_iniciales().items()}
    contador = [0]

    def bt(asig, i):
        if i == len(ORDEN):
            return
        t = ORDEN[i]
        for v in dom[t]:
            contador[0] += 1
            if consistente(asig, t, v):
                asig[t] = v
                bt(asig, i + 1)
                del asig[t]

    bt({}, 0)
    return contador[0]

### Salida

In [11]:
def mostrar_horario(asig, prio):
    print(f"Mejor horario (prioridad total = {prio}):")
    for t in ORDEN:
        v = asig[t]
        if v is NO_PROGRAMADA:
            print(f"  Tarea {t}: Ø (no programada)")
        else:
            d, ini = v
            print(f"  Tarea {t}: {DIAS[d]} {hora(ini)} - {hora(ini + TAREAS[t]['dur'])}")


def graficar(nodos_fb, nodos_opt):
    etiquetas = ["Fuerza Bruta", "VMR + VMRe + CD"]
    valores = [nodos_fb, nodos_opt]
    fig, ax = plt.subplots(figsize=(6, 4))
    barras = ax.bar(etiquetas, valores, color=["#c0392b", "#27ae60"])
    ax.bar_label(barras, padding=3)
    ax.set_ylabel("Nodos evaluados")
    ax.set_title("Kairos (PSR): Fuerza Bruta vs. heurísticas")
    fig.tight_layout()
    fig.savefig("comparacion_nodos.png", dpi=150)
    plt.show()


if __name__ == "__main__":
    busqueda = BusquedaOptimizada()
    busqueda.resolver()

    if nodos_fuerza_bruta is None:
        nodos_fuerza_bruta = fuerza_bruta()

    mostrar_horario(busqueda.mejor_asig, busqueda.mejor_prio)
    print(f"\nNodos evaluados (VMR + VMRe + CD): {busqueda.nodos}")
    print(f"Nodos evaluados (Fuerza Bruta):    {nodos_fuerza_bruta}")
    print(f"Dominios vaciados detectados por CD: {busqueda.vacios_detectados}")
    ahorro = 100 * (1 - busqueda.nodos / nodos_fuerza_bruta)
    print(f"Reducción de nodos: {ahorro:.1f}%")

    graficar(nodos_fuerza_bruta, busqueda.nodos)

Mejor horario (prioridad total = 8):
  Tarea A: Martes 18:00 - 21:00
  Tarea B: Lunes 18:00 - 21:00
  Tarea C: Ø (no programada)

Nodos evaluados (VMR + VMRe + CD): 119
Nodos evaluados (Fuerza Bruta):    320
Dominios vaciados detectados por CD: 9
Reducción de nodos: 62.8%


**Resultado:** con VMR, VMRe y CD se pasa de 320 a 119 nodos, una reducción de 62,8 %. La CD detectó 9 veces que una tarea se quedaba sin bloques y cortó ese camino.

## Fase 3: caso realista con bloques fijos y poda optimista
El caso crece a tres días y cinco tareas, con clases como bloques fijos (R1). Además de las heurísticas se agrega la poda: si la prioridad acumulada más lo que aún se podría sumar no supera al mejor horario encontrado, esa rama se descarta.

In [12]:
DIAS = ["Lunes", "Martes", "Miércoles"]
INICIO_JORNADA = 14 * 60
FIN_JORNADA = 22 * 60
BLOQUE_MIN = 30
BLOQUES_POR_DIA = (FIN_JORNADA - INICIO_JORNADA) // BLOQUE_MIN
TOPE_DIARIO_BLOQUES = 6 * 2
NO_PROGRAMADA = None

EXIGIR_DESCANSO_EN_DOMINIO = True


def a_bloque(hhmm):
    """'16:00' -> índice de bloque dentro de la jornada (0 = 14:00)."""
    h, m = map(int, hhmm.split(":"))
    return (h * 60 + m - INICIO_JORNADA) // BLOQUE_MIN


def hora(bloque):
    m = INICIO_JORNADA + bloque * BLOQUE_MIN
    return f"{m // 60:02d}:{m % 60:02d}"

### Datos del caso de prueba

In [13]:
FIJOS_ENTRADA = [
    (0, "14:00", "16:00"),
    (1, "18:00", "20:00"),
]

TAREAS = {
    "T1": {"nombre": "Ensayo",     "dur": 8, "prio": 5, "vence": (2, 16)},
    "T2": {"nombre": "Ejercicios", "dur": 4, "prio": 4, "vence": (0, 16)},
    "T3": {"nombre": "Lectura",    "dur": 2, "prio": 2, "vence": (1, 16)},
    "T4": {"nombre": "Proyecto",   "dur": 6, "prio": 5, "vence": (2, 16)},
    "T5": {"nombre": "Foro",       "dur": 2, "prio": 1, "vence": (0, 16)},
}
ORDEN = list(TAREAS)

### R1: dominios iniciales con bloques ocupados

In [14]:
def construir_ocupados(fijos):
    """Devuelve {día: set(bloques ocupados)}."""
    ocupados = {d: set() for d in range(len(DIAS))}
    for d, ini, fin in fijos:
        ocupados[d].update(range(a_bloque(ini), a_bloque(fin)))
    return ocupados


def dominios_iniciales(ocupados):
    """Para cada tarea, los bloques de inicio (día, bloque) tales que:
       - la tarea (y su descanso posterior) cabe continua y libre de bloques fijos,
       - no se sale de la jornada (no pasa a otro día),
       - termina antes de su fecha límite.
       'Ø' no se guarda: siempre está disponible y se prueba de último."""
    dominios = {}
    for t, info in TAREAS.items():
        dia_v, blq_v = info["vence"]
        dom = []
        for d in range(len(DIAS)):
            for ini in range(BLOQUES_POR_DIA):
                fin = ini + info["dur"]
                if fin > BLOQUES_POR_DIA:
                    continue
                if d > dia_v or (d == dia_v and fin > blq_v):
                    continue
                hasta = fin + 1 if EXIGIR_DESCANSO_EN_DOMINIO else fin
                hasta = min(hasta, BLOQUES_POR_DIA)
                if any(b in ocupados[d] for b in range(ini, hasta)):
                    continue
                dom.append((d, ini))
        dominios[t] = dom
    return dominios

### R2 + R4

In [15]:
def consistente(asignacion, tarea, valor):
    """R2: sin cruce y 1 bloque de descanso entre tareas del mismo día.
       R4: tope diario de horas."""
    if valor is NO_PROGRAMADA:
        return True
    dia, ini = valor
    fin = ini + TAREAS[tarea]["dur"]
    bloques_dia = TAREAS[tarea]["dur"]
    for otra, v in asignacion.items():
        if v is NO_PROGRAMADA or v[0] != dia:
            continue
        ini2 = v[1]
        fin2 = ini2 + TAREAS[otra]["dur"]
        bloques_dia += TAREAS[otra]["dur"]
        if not (ini >= fin2 + 1 or ini2 >= fin + 1):
            return False
    return bloques_dia <= TOPE_DIARIO_BLOQUES


def prioridad(asignacion):
    return sum(TAREAS[t]["prio"] for t, v in asignacion.items() if v is not NO_PROGRAMADA)

### Heurísticas (VMR, VMRe, CD)

In [16]:
def comprobacion_directa(dominios, asignacion, tarea, valor):
    """CD: recorta de los dominios pendientes lo que choca con la nueva asignación."""
    nueva = dict(asignacion)
    nueva[tarea] = valor
    return {t: [v for v in dom if consistente(nueva, t, v)]
            for t, dom in dominios.items() if t != tarea}


def seleccionar_variable_vmr(dominios):
    """VMR: la tarea pendiente con menos bloques legales (empate: orden de ORDEN)."""
    return min(dominios, key=lambda t: (len(dominios[t]), ORDEN.index(t)))


def ordenar_valores_vmre(tarea, dominios, asignacion):
    """VMRe: primero los bloques que dejan más opciones a las demás; 'Ø' al final."""
    puntuados = []
    for v in dominios[tarea]:
        restantes = comprobacion_directa(dominios, asignacion, tarea, v)
        puntuados.append((sum(len(d) for d in restantes.values()), v))
    puntuados.sort(key=lambda x: -x[0])
    return [v for _, v in puntuados] + [NO_PROGRAMADA]

### Búsqueda con poda optimista

In [17]:
class Busqueda:
    def __init__(self, ocupados, usar_poda=True):
        self.ocupados = ocupados
        self.usar_poda = usar_poda
        self.nodos = 0
        self.podas = 0
        self.primera_poda_en_nodo = None
        self.mejor_prio = -1
        self.mejor_asig = None

    def resolver(self):
        self.backtrack({}, dominios_iniciales(self.ocupados))

    def backtrack(self, asignacion, dominios):
        acumulada = prioridad(asignacion)

        if not dominios:
            if acumulada > self.mejor_prio:
                self.mejor_prio, self.mejor_asig = acumulada, dict(asignacion)
            return

        if self.usar_poda:
            cota = acumulada + sum(TAREAS[t]["prio"] for t, d in dominios.items() if d)
            if cota <= self.mejor_prio:
                self.podas += 1
                if self.primera_poda_en_nodo is None:
                    self.primera_poda_en_nodo = self.nodos
                return

        tarea = seleccionar_variable_vmr(dominios)
        valores = ordenar_valores_vmre(tarea, dominios, asignacion)

        for valor in valores:
            self.nodos += 1
            if valor is NO_PROGRAMADA:
                nuevos = {t: d for t, d in dominios.items() if t != tarea}
            else:
                nuevos = comprobacion_directa(dominios, asignacion, tarea, valor)
            asignacion[tarea] = valor
            self.backtrack(asignacion, nuevos)
            del asignacion[tarea]

### Salida

In [18]:
def mostrar_horario(asig, prio):
    print(f"Mejor horario (prioridad total = {prio}):")
    filas = []
    for d, ini, fin in FIJOS_ENTRADA:
        filas.append((d, a_bloque(ini), f"[FIJO] {ini} - {fin}"))
    for t in ORDEN:
        v = asig[t]
        if v is not NO_PROGRAMADA:
            d, ini = v
            filas.append((d, ini, f"{t} {TAREAS[t]['nombre']}: "
                                  f"{hora(ini)} - {hora(ini + TAREAS[t]['dur'])}"))
    for d, _, texto in sorted(filas):
        print(f"  {DIAS[d]:<9} {texto}")
    for t in ORDEN:
        if asig[t] is NO_PROGRAMADA:
            print(f"  {t} {TAREAS[t]['nombre']}: Ø (no programada)")


if __name__ == "__main__":
    ocupados = construir_ocupados(FIJOS_ENTRADA)

    print("Dominios iniciales (R1 aplicada):")
    for t, dom in dominios_iniciales(ocupados).items():
        print(f"  {t}: {len(dom)} bloques de inicio")

    con = Busqueda(ocupados, usar_poda=True)
    con.resolver()
    sin = Busqueda(ocupados, usar_poda=False)
    sin.resolver()

    print()
    mostrar_horario(con.mejor_asig, con.mejor_prio)
    print(f"\nNodos evaluados CON poda: {con.nodos}")
    print(f"Nodos evaluados SIN poda: {sin.nodos}")
    if con.podas:
        print(f"¿Se activó la poda? SÍ: {con.podas} cortes "
              f"(primero tras {con.primera_poda_en_nodo} nodos)")
    else:
        print("¿Se activó la poda? NO")
    print(f"Prioridad óptima coincide sin/con poda: {con.mejor_prio == sin.mejor_prio}")

Dominios iniciales (R1 aplicada):
  T1: 14 bloques de inicio
  T2: 9 bloques de inicio
  T3: 20 bloques de inicio
  T4: 20 bloques de inicio
  T5: 11 bloques de inicio

Mejor horario (prioridad total = 17):
  Lunes     [FIJO] 14:00 - 16:00
  Lunes     T2 Ejercicios: 16:00 - 18:00
  Lunes     T5 Foro: 18:30 - 19:30
  Lunes     T3 Lectura: 20:00 - 21:00
  Martes    T4 Proyecto: 14:00 - 17:00
  Martes    [FIJO] 18:00 - 20:00
  Miércoles T1 Ensayo: 14:00 - 18:00

Nodos evaluados CON poda: 37
Nodos evaluados SIN poda: 34542
¿Se activó la poda? SÍ: 26 cortes (primero tras 12 nodos)
Prioridad óptima coincide sin/con poda: True


**Resultado:** el mejor horario suma prioridad 17. Con poda se evalúan 37 nodos y sin poda 34.542, con la misma prioridad óptima.

## Fase 4: prueba de estrés y alertas
Se suman dos tareas (siete en total) para provocar una sobrecarga. Cuando una tarea queda sin programar, el módulo de alertas muestra un mensaje según su prioridad, y se calcula qué porcentaje de la prioridad total quedó cubierto.

In [19]:
import matplotlib
matplotlib.use("Agg") if __import__("os").environ.get("KAIROS_NOSHOW") else None
import matplotlib.pyplot as plt

### Configuración del calendario

In [20]:
DIAS = ["Lunes", "Martes", "Miércoles"]
INICIO_JORNADA = 14 * 60
FIN_JORNADA = 22 * 60
BLOQUE_MIN = 30
BLOQUES_POR_DIA = (FIN_JORNADA - INICIO_JORNADA) // BLOQUE_MIN
TOPE_DIARIO_BLOQUES = 6 * 2
NO_PROGRAMADA = None

EXIGIR_DESCANSO_EN_DOMINIO = True


def a_bloque(hhmm):
    """'16:00' -> índice de bloque dentro de la jornada (0 = 14:00)."""
    h, m = map(int, hhmm.split(":"))
    return (h * 60 + m - INICIO_JORNADA) // BLOQUE_MIN


def hora(bloque):
    m = INICIO_JORNADA + bloque * BLOQUE_MIN
    return f"{m // 60:02d}:{m % 60:02d}"

### Datos del caso de prueba

In [21]:
FIJOS_ENTRADA = [
    (0, "14:00", "16:00"),
    (1, "18:00", "20:00"),
]

TAREAS = {
    "T1": {"nombre": "Ensayo",     "dur": 8, "prio": 5, "vence": (2, 16)},
    "T2": {"nombre": "Ejercicios", "dur": 4, "prio": 4, "vence": (0, 16)},
    "T3": {"nombre": "Lectura",    "dur": 2, "prio": 2, "vence": (1, 16)},
    "T4": {"nombre": "Proyecto",   "dur": 6, "prio": 5, "vence": (2, 16)},
    "T5": {"nombre": "Foro",       "dur": 2, "prio": 1, "vence": (0, 16)},
    "T6": {"nombre": "Estudio Parcial", "dur": 8, "prio": 5, "vence": (1, 16)},
    "T7": {"nombre": "Video",          "dur": 6, "prio": 2, "vence": (0, 16)},
}
ORDEN = list(TAREAS)

### R1: dominios iniciales con bloques ocupados

In [22]:
def construir_ocupados(fijos):
    """Devuelve {día: set(bloques ocupados)}."""
    ocupados = {d: set() for d in range(len(DIAS))}
    for d, ini, fin in fijos:
        ocupados[d].update(range(a_bloque(ini), a_bloque(fin)))
    return ocupados


def dominios_iniciales(ocupados):
    """Para cada tarea, los bloques de inicio (día, bloque) tales que:
       - la tarea (y su descanso posterior) cabe continua y libre de bloques fijos,
       - no se sale de la jornada (no pasa a otro día),
       - termina antes de su fecha límite.
       'Ø' no se guarda: siempre está disponible y se prueba de último."""
    dominios = {}
    for t, info in TAREAS.items():
        dia_v, blq_v = info["vence"]
        dom = []
        for d in range(len(DIAS)):
            for ini in range(BLOQUES_POR_DIA):
                fin = ini + info["dur"]
                if fin > BLOQUES_POR_DIA:
                    continue
                if d > dia_v or (d == dia_v and fin > blq_v):
                    continue
                hasta = fin + 1 if EXIGIR_DESCANSO_EN_DOMINIO else fin
                hasta = min(hasta, BLOQUES_POR_DIA)
                if any(b in ocupados[d] for b in range(ini, hasta)):
                    continue
                dom.append((d, ini))
        dominios[t] = dom
    return dominios

### R2 + R4

In [23]:
def consistente(asignacion, tarea, valor):
    """R2: sin cruce y 1 bloque de descanso entre tareas del mismo día.
       R4: tope diario de horas."""
    if valor is NO_PROGRAMADA:
        return True
    dia, ini = valor
    fin = ini + TAREAS[tarea]["dur"]
    bloques_dia = TAREAS[tarea]["dur"]
    for otra, v in asignacion.items():
        if v is NO_PROGRAMADA or v[0] != dia:
            continue
        ini2 = v[1]
        fin2 = ini2 + TAREAS[otra]["dur"]
        bloques_dia += TAREAS[otra]["dur"]
        if not (ini >= fin2 + 1 or ini2 >= fin + 1):
            return False
    return bloques_dia <= TOPE_DIARIO_BLOQUES


def prioridad(asignacion):
    return sum(TAREAS[t]["prio"] for t, v in asignacion.items() if v is not NO_PROGRAMADA)

### Heurísticas (VMR, VMRe, CD)

In [24]:
def comprobacion_directa(dominios, asignacion, tarea, valor):
    """CD: recorta de los dominios pendientes lo que choca con la nueva asignación."""
    nueva = dict(asignacion)
    nueva[tarea] = valor
    return {t: [v for v in dom if consistente(nueva, t, v)]
            for t, dom in dominios.items() if t != tarea}


def seleccionar_variable_vmr(dominios):
    """VMR: la tarea pendiente con menos bloques legales (empate: orden de ORDEN)."""
    return min(dominios, key=lambda t: (len(dominios[t]), ORDEN.index(t)))


def ordenar_valores_vmre(tarea, dominios, asignacion):
    """VMRe: primero los bloques que dejan más opciones a las demás; 'Ø' al final."""
    puntuados = []
    for v in dominios[tarea]:
        restantes = comprobacion_directa(dominios, asignacion, tarea, v)
        puntuados.append((sum(len(d) for d in restantes.values()), v))
    puntuados.sort(key=lambda x: -x[0])
    return [v for _, v in puntuados] + [NO_PROGRAMADA]

### Búsqueda con poda optimista

In [25]:
class Busqueda:
    def __init__(self, ocupados, usar_poda=True):
        self.ocupados = ocupados
        self.usar_poda = usar_poda
        self.nodos = 0
        self.podas = 0
        self.primera_poda_en_nodo = None
        self.mejor_prio = -1
        self.mejor_asig = None

    def resolver(self):
        self.backtrack({}, dominios_iniciales(self.ocupados))

    def backtrack(self, asignacion, dominios):
        acumulada = prioridad(asignacion)

        if not dominios:
            if acumulada > self.mejor_prio:
                self.mejor_prio, self.mejor_asig = acumulada, dict(asignacion)
            return

        if self.usar_poda:
            cota = acumulada + sum(TAREAS[t]["prio"] for t, d in dominios.items() if d)
            if cota <= self.mejor_prio:
                self.podas += 1
                if self.primera_poda_en_nodo is None:
                    self.primera_poda_en_nodo = self.nodos
                return

        tarea = seleccionar_variable_vmr(dominios)
        valores = ordenar_valores_vmre(tarea, dominios, asignacion)

        for valor in valores:
            self.nodos += 1
            if valor is NO_PROGRAMADA:
                nuevos = {t: d for t, d in dominios.items() if t != tarea}
            else:
                nuevos = comprobacion_directa(dominios, asignacion, tarea, valor)
            asignacion[tarea] = valor
            self.backtrack(asignacion, nuevos)
            del asignacion[tarea]

### Salida

In [26]:
def mostrar_horario(asig, prio):
    print(f"Mejor horario (prioridad total = {prio}):")
    filas = []
    for d, ini, fin in FIJOS_ENTRADA:
        filas.append((d, a_bloque(ini), f"[FIJO] {ini} - {fin}"))
    for t in ORDEN:
        v = asig[t]
        if v is not NO_PROGRAMADA:
            d, ini = v
            filas.append((d, ini, f"{t} {TAREAS[t]['nombre']}: "
                                  f"{hora(ini)} - {hora(ini + TAREAS[t]['dur'])}"))
    for d, _, texto in sorted(filas):
        print(f"  {DIAS[d]:<9} {texto}")
    for t in ORDEN:
        if asig[t] is NO_PROGRAMADA:
            print(f"  {t} {TAREAS[t]['nombre']}: Ø (no programada)")

### Módulo de salida y alertas

In [27]:
MSG_CRITICA = ("Alerta: Tarea crítica omitida. Sugerencia: Libera bloques fijos "
               "de tu calendario para hacerle espacio.")
MSG_BAJA = ("Sugerencia: Considera bajar su prioridad, reducir su duración "
            "o pedir más plazo.")
MSG_MEDIA = ("Aviso: Tarea de prioridad media omitida. Sugerencia: Revisa si puedes "
             "moverla a otro día o ampliar su plazo.")


def generar_alertas(mejor_horario):
    """Recorre el mejor horario y devuelve un mensaje por cada tarea en 'Ø'."""
    alertas = []
    for t in ORDEN:
        if mejor_horario[t] is not NO_PROGRAMADA:
            continue
        info = TAREAS[t]
        if info["prio"] >= 4:
            msg = MSG_CRITICA
        elif info["prio"] <= 2:
            msg = MSG_BAJA
        else:
            msg = MSG_MEDIA
        alertas.append(f"[{t} {info['nombre']} | prioridad {info['prio']}] {msg}")
    return alertas


def prioridad_cubierta(prio_plan):
    """(Prioridad del plan / suma de todas las prioridades) * 100."""
    total = sum(info["prio"] for info in TAREAS.values())
    return prio_plan / total * 100, total


def graficar(nodos_sin, nodos_con):
    etiquetas = ["Sin poda\n(VMR+VMRe+CD)", "Con poda\n(+ cota optimista)"]
    fig, ax = plt.subplots(figsize=(6, 4))
    barras = ax.bar(etiquetas, [nodos_sin, nodos_con], color=["#c0392b", "#27ae60"])
    ax.bar_label(barras, padding=3)
    ax.set_yscale("log")
    ax.set_ylabel("Nodos evaluados (escala log)")
    ax.set_title("Kairos (PSR): efecto de la poda optimista")
    fig.tight_layout()
    fig.savefig("comparacion_poda.png", dpi=150)
    plt.show()


if __name__ == "__main__":
    ocupados = construir_ocupados(FIJOS_ENTRADA)

    con = Busqueda(ocupados, usar_poda=True)
    con.resolver()
    sin = Busqueda(ocupados, usar_poda=False)
    sin.resolver()

    print("=== HORARIO FINAL ===")
    mostrar_horario(con.mejor_asig, con.mejor_prio)

    pct, total = prioridad_cubierta(con.mejor_prio)
    print(f"\nPrioridad cubierta: {con.mejor_prio}/{total} = {pct:.1f}%")
    print(f"Nodos evaluados CON poda: {con.nodos}")
    print(f"Nodos evaluados SIN poda: {sin.nodos}")
    print(f"Poda activada: {con.podas} cortes (primero tras {con.primera_poda_en_nodo} nodos)")
    print(f"Misma prioridad óptima sin/con poda: {con.mejor_prio == sin.mejor_prio}")

    print("\n=== ALERTAS ===")
    alertas = generar_alertas(con.mejor_asig)
    if alertas:
        for a in alertas:
            print(" -", a)
    else:
        print(" Todas las tareas quedaron programadas.")

    graficar(sin.nodos, con.nodos)

=== HORARIO FINAL ===
Mejor horario (prioridad total = 18):
  Lunes     [FIJO] 14:00 - 16:00
  Lunes     T6 Estudio Parcial: 16:00 - 20:00
  Lunes     T5 Foro: 20:30 - 21:30
  Martes    T4 Proyecto: 14:00 - 17:00
  Martes    [FIJO] 18:00 - 20:00
  Martes    T3 Lectura: 20:00 - 21:00
  Miércoles T1 Ensayo: 14:00 - 18:00
  T2 Ejercicios: Ø (no programada)
  T7 Video: Ø (no programada)

Prioridad cubierta: 18/24 = 75.0%
Nodos evaluados CON poda: 46
Nodos evaluados SIN poda: 50795
Poda activada: 31 cortes (primero tras 11 nodos)
Misma prioridad óptima sin/con poda: True

=== ALERTAS ===
 - [T2 Ejercicios | prioridad 4] Alerta: Tarea crítica omitida. Sugerencia: Libera bloques fijos de tu calendario para hacerle espacio.
 - [T7 Video | prioridad 2] Sugerencia: Considera bajar su prioridad, reducir su duración o pedir más plazo.


**Resultado:** se programan 5 de las 7 tareas (prioridad 18 de 24, 75 %) con 46 nodos con poda frente a 50.795 sin poda. Las dos tareas omitidas generan su alerta.